# 🎯 Artemisia v9 — Backtest Complet Multi-Stratégies
## Hyperliquid Perpetual Futures

**Objectif :** Diagnostiquer les résultats calamiteux en production et valider les nouveaux paramètres.

**Exchange :** Hyperliquid Perps — Taker = 3.5 bps, Maker = -1 bp, Slippage ~2 bps/side

**Stratégies :** S8EMS · MomentumLS · BreakoutControlled · MeanReversionKalman · DonchianTrend · RSIBollingerReversion

---
| Section | Contenu |
|---------|--------|
| 0 | Requirements & Imports |
| 1 | Téléchargement des données |
| 2 | Modèle de coûts — Pourquoi les résultats étaient nuls |
| 3 | Backtesteur vectorisé (moteur) |
| 4 | Backtest S8EMS |
| 5 | Backtest MomentumLS |
| 6 | Backtest DonchianTrend |
| 7 | Backtest RSIBollingerReversion |
| 8 | Backtest MeanReversionKalman |
| 9 | Dashboard comparatif |
| 10 | Walk-Forward Validation |
| 11 | Critère de Kelly |
| 12 | Conclusions & Plan d'action |


## 0. Requirements & Imports

In [ ]:
# Installation des dépendances (à exécuter une seule fois)
import subprocess, sys
pkgs = ['ccxt', 'pandas', 'numpy', 'plotly', 'scipy', 'requests', 'pykalman']
for pkg in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('✅ Toutes les dépendances installées')


In [ ]:
import os, json, time, warnings
import numpy as np
import pandas as pd
from datetime import datetime, timedelta, timezone
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
import scipy.stats as stats

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio

import ccxt
warnings.filterwarnings('ignore')

# Thème sombre global (cohérent avec la GUI Artemisia)
pio.templates.default = 'plotly_dark'
DARK_BG   = '#0d1117'
ACCENT    = '#00d4aa'
RED       = '#ff4b4b'
YELLOW    = '#ffd700'

# Dossier de données
DATA_DIR = Path(r'C:\Users\jeanb\Documents\Mercantour\Arbitragedetector\genepi\gptversion\artemisia\hyperliquidbis\artemisia_v9\backtest\data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Paramètres Hyperliquid
TAKER_FEE   = 0.00035   # 3.5 bps
MAKER_FEE   = -0.0001   # -1 bp (rebate)
SLIPPAGE_BPS = 2.0      # bps par côté

SYMBOLS = ['BTC/USDT', 'ETH/USDT', 'SOL/USDT', 'AVAX/USDT', 'LINK/USDT', 'ARB/USDT']
SHORT_NAMES = [s.split('/')[0] for s in SYMBOLS]
CAPITAL_PER_STRAT = 500.0

print('✅ Imports OK')
print(f'Exchange fees: Taker={TAKER_FEE*10000:.1f}bps  Maker={MAKER_FEE*10000:.1f}bps  Slippage={SLIPPAGE_BPS}bps/side')


## 1. Téléchargement des données OHLCV

180 jours de données 1h et 15m pour les 6 symboles principaux, via Binance (ccxt).
Les données sont mises en cache dans `backtest/data/` pour éviter les re-téléchargements.


In [ ]:
def fetch_ohlcv(symbol: str, timeframe: str = '1h', days: int = 180,
                exchange_id: str = 'binance', force_refresh: bool = False) -> pd.DataFrame:
    """
    Télécharge les données OHLCV depuis Binance via ccxt.
    Mise en cache automatique dans DATA_DIR.
    """
    safe_sym = symbol.replace('/', '_')
    cache_path = DATA_DIR / f'{safe_sym}_{timeframe}_{days}d.csv'

    if cache_path.exists() and not force_refresh:
        df = pd.read_csv(cache_path, index_col=0, parse_dates=True)
        age_h = (datetime.now() - datetime.fromtimestamp(cache_path.stat().st_mtime)).total_seconds() / 3600
        print(f'  📂 Cache {symbol} {timeframe}: {len(df)} barres (âge: {age_h:.1f}h)')
        return df

    exchange = ccxt.binance({'enableRateLimit': True})
    since_ms  = int((datetime.now(timezone.utc) - timedelta(days=days)).timestamp() * 1000)
    all_bars  = []

    tf_ms_map = {'1m': 60000, '5m': 300000, '15m': 900000, '1h': 3600000,
                 '4h': 14400000, '1d': 86400000}
    tf_ms = tf_ms_map.get(timeframe, 3600000)
    limit  = 1000

    cursor = since_ms
    while cursor < int(datetime.now(timezone.utc).timestamp() * 1000):
        bars = exchange.fetch_ohlcv(symbol, timeframe, since=cursor, limit=limit)
        if not bars:
            break
        all_bars.extend(bars)
        cursor = bars[-1][0] + tf_ms
        time.sleep(exchange.rateLimit / 1000)

    df = pd.DataFrame(all_bars, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)
    df.set_index('timestamp', inplace=True)
    df = df[~df.index.duplicated(keep='last')].sort_index()
    df.to_csv(cache_path)
    print(f'  ✅ Téléchargé {symbol} {timeframe}: {len(df)} barres → {cache_path.name}')
    return df


def fetch_all_symbols(symbols: list, timeframes: list = ['1h', '15m'],
                      days: int = 180, force_refresh: bool = False) -> dict:
    """Télécharge toutes les données et retourne un dict imbriqué {tf: {sym: df}}."""
    data = {}
    for tf in timeframes:
        data[tf] = {}
        print(f'\n⏬ Timeframe {tf}:')
        for sym in symbols:
            try:
                data[tf][sym.split('/')[0]] = fetch_ohlcv(sym, tf, days, force_refresh=force_refresh)
            except Exception as e:
                print(f'  ⚠️  {sym} {tf}: {e}')
    return data


print('Fonctions de téléchargement définies.')
print('Lancement du téléchargement (peut prendre 2-5 min si pas de cache)...')


In [ ]:
# Téléchargement effectif (décommenter force_refresh=True pour forcer)
RAW = fetch_all_symbols(SYMBOLS, timeframes=['1h', '15m'], days=180, force_refresh=False)

print('\n📊 Résumé des données:')
for tf, syms in RAW.items():
    for sym, df in syms.items():
        pct_nan = df.isnull().mean().mean() * 100
        print(f'  {sym:6s} {tf:4s}: {len(df):5d} barres | '
              f'{df.index[0].date()} → {df.index[-1].date()} | NaN={pct_nan:.2f}%')


In [ ]:
# Contrôle qualité — Détection de gaps
print('\n🔍 Contrôle qualité — Gaps détectés:\n')
for tf, syms in RAW.items():
    tf_td = pd.Timedelta('1h') if tf == '1h' else pd.Timedelta('15min')
    for sym, df in syms.items():
        gaps = df.index.to_series().diff().dropna()
        big  = gaps[gaps > tf_td * 1.5]
        if len(big) > 0:
            print(f'  ⚠️  {sym} {tf}: {len(big)} gaps (max={big.max()})')
        else:
            print(f'  ✅ {sym} {tf}: aucun gap')

# Aperçu BTC 1h
btc_1h = RAW['1h']['BTC']
print('\nAperçu BTC/USDT 1h:')
btc_1h.tail(5)


## 2. Modèle de coûts — Pourquoi les résultats étaient calamiteux

### Le diagnostic mathématique

**Hyperliquid fees (perps) :**
- Taker : **3.5 bps** (0.035%)
- Maker : **-1 bp** (rebate)
- Slippage estimé : **~2 bps par côté**

**Round-trip minimum (taker+taker) :** 2×3.5 + 2×2 = **11 bps**

**Le bug S8EMS :** `position=4% × $100 = $4`, `spread=1.5bps`
- Gain espéré : `$4 × 1.5bps = $0.0006`
- Coût réel : `$4 × 11bps = $0.0044`
- **Résultat : -$0.0038 à chaque trade → spirale de mort**


In [ ]:
class HyperliquidFeeModel:
    """Modèle de coûts complet pour Hyperliquid Perpetuals."""

    TAKER = 0.00035    # 3.5 bps
    MAKER = -0.0001    # -1 bp (rebate)
    SLIP  = 0.0002     # 2 bps par côté (estimé)

    def round_trip_cost_bps(self, entry_type='taker', exit_type='taker') -> float:
        """Coût total aller-retour en bps."""
        entry_fee = self.TAKER if entry_type == 'taker' else self.MAKER
        exit_fee  = self.TAKER if exit_type  == 'taker' else self.MAKER
        return (entry_fee + exit_fee + 2 * self.SLIP) * 10_000

    def trade_pnl_usd(self, position_usd: float, spread_bps: float,
                       entry_type='taker', exit_type='taker') -> float:
        """PnL net en USD pour un trade market-making (spread capturé)."""
        gross     = position_usd * spread_bps / 10_000
        cost      = position_usd * self.round_trip_cost_bps(entry_type, exit_type) / 10_000
        return gross - cost

    def breakeven_spread_bps(self, entry_type='taker', exit_type='taker') -> float:
        """Spread minimum pour être profitable."""
        return self.round_trip_cost_bps(entry_type, exit_type)

    def cost_per_trade_usd(self, position_usd: float, entry_type='taker', exit_type='taker') -> float:
        return position_usd * self.round_trip_cost_bps(entry_type, exit_type) / 10_000


fm = HyperliquidFeeModel()

print('=== DIAGNOSTIC S8EMS — ANCIENS PARAMÈTRES ===')
print(f"Round-trip taker+taker : {fm.round_trip_cost_bps('taker','taker'):.1f} bps")
print(f"Round-trip maker+maker : {fm.round_trip_cost_bps('maker','maker'):.1f} bps")
print()

# Anciens params
pos_old    = 4.0    # $4 (4% × $100)
spread_old = 1.5    # 1.5 bps
pnl_old    = fm.trade_pnl_usd(pos_old, spread_old)
print(f'ANCIEN : position=${pos_old}, spread={spread_old}bps')
print(f'  Gain brut     : ${pos_old * spread_old / 10000:.6f}')
print(f'  Coût aller-retour: ${fm.cost_per_trade_usd(pos_old):.6f}')
print(f'  PnL net       : ${pnl_old:.6f}  ← TOUJOURS NÉGATIF')
print()

# Nouveaux params
pos_new    = 100.0  # $100 (20% × $500)
spread_new = 8.0    # 8 bps
pnl_new    = fm.trade_pnl_usd(pos_new, spread_new)
print(f'NOUVEAU : position=${pos_new}, spread={spread_new}bps')
print(f'  Gain brut     : ${pos_new * spread_new / 10000:.4f}')
print(f'  Coût aller-retour: ${fm.cost_per_trade_usd(pos_new):.4f}')
print(f'  PnL net       : ${pnl_new:.4f}  ← PROFITABLE')
print()
print(f"Break-even spread : {fm.breakeven_spread_bps():.1f} bps (seuil minimal absolu)")


In [ ]:
# Tableau coût vs (position, spread)
positions = [4, 10, 25, 50, 100, 150, 200]
spreads   = [1.5, 3, 5, 8, 10, 15, 20]

matrix = pd.DataFrame(index=[f'${p}' for p in positions],
                       columns=[f'{s}bps' for s in spreads])
for p in positions:
    for s in spreads:
        pnl = fm.trade_pnl_usd(p, s)
        matrix.loc[f'${p}', f'{s}bps'] = f'{pnl:+.4f}'

print('PnL net par trade ($) — Position (lignes) × Spread (colonnes):')
print('Valeurs positives = profitable, négatives = perte systématique')
matrix


In [ ]:
# Graphique interactif : PnL vs spread pour différentes tailles de position
spread_range = np.linspace(0, 25, 300)
fig = go.Figure()

colors_pos = px.colors.sequential.Plasma
for i, pos in enumerate([4, 25, 50, 100, 200]):
    pnls = [fm.trade_pnl_usd(pos, s) for s in spread_range]
    c    = colors_pos[int(i * len(colors_pos) / 5)]
    fig.add_trace(go.Scatter(
        x=spread_range, y=pnls, name=f'Position ${pos}',
        line=dict(color=c, width=2)
    ))

be = fm.breakeven_spread_bps()
fig.add_vline(x=be, line_dash='dash', line_color=YELLOW,
              annotation_text=f'Break-even = {be:.1f}bps', annotation_position='top right')
fig.add_hrect(y0=-0.5, y1=0, fillcolor=RED, opacity=0.1, line_width=0)
fig.add_annotation(x=2, y=-0.25, text='ZONE PERDANTE', font=dict(color=RED, size=12))

# Anciens params
fig.add_trace(go.Scatter(x=[1.5], y=[fm.trade_pnl_usd(4, 1.5)],
    mode='markers', marker=dict(size=15, color=RED, symbol='x'),
    name='⚠️ Ancien S8EMS ($4, 1.5bps)'))

# Nouveaux params
fig.add_trace(go.Scatter(x=[8.0], y=[fm.trade_pnl_usd(100, 8.0)],
    mode='markers', marker=dict(size=15, color=ACCENT, symbol='star'),
    name='✅ Nouveau S8EMS ($100, 8bps)'))

fig.update_layout(
    title='PnL net par trade vs Spread capturé (Hyperliquid Taker+Taker)',
    xaxis_title='Spread capturé (bps)', yaxis_title='PnL net ($)',
    height=500, template='plotly_dark',
    legend=dict(x=0.01, y=0.99)
)
fig.show()


## 3. Backtesteur vectorisé

Moteur de backtest vectorisé avec calcul complet des frais, drawdown, métriques risk-adjusted.


In [ ]:
@dataclass
class BacktestResult:
    trades: pd.DataFrame
    equity: pd.Series
    capital: float = 500.0
    strategy_name: str = 'Strategy'

    def summary(self) -> dict:
        eq   = self.equity
        trd  = self.trades
        wins = trd[trd['pnl_net'] > 0]
        loss = trd[trd['pnl_net'] <= 0]
        return {
            'strategy':      self.strategy_name,
            'n_trades':      len(trd),
            'win_rate':      self.win_rate(),
            'total_pnl':     trd['pnl_net'].sum(),
            'avg_trade_pnl': trd['pnl_net'].mean() if len(trd) else 0,
            'profit_factor': self.profit_factor(),
            'sharpe':        self.sharpe(),
            'sortino':       self.sortino(),
            'calmar':        self.calmar(),
            'max_drawdown':  self.max_drawdown(),
            'final_equity':  eq.iloc[-1] if len(eq) else self.capital,
            'total_return':  (eq.iloc[-1] / self.capital - 1) * 100 if len(eq) else 0,
        }

    def equity_curve(self) -> pd.Series: return self.equity
    def trade_log(self)    -> pd.DataFrame: return self.trades

    def _returns(self) -> pd.Series:
        eq = self.equity
        if len(eq) < 2: return pd.Series(dtype=float)
        return eq.pct_change().dropna()

    def sharpe(self, risk_free: float = 0.05, periods_per_year: int = 8760) -> float:
        r = self._returns()
        if len(r) < 2 or r.std() == 0: return 0.0
        rf_per_bar = risk_free / periods_per_year
        excess = r - rf_per_bar
        return float(np.sqrt(periods_per_year) * excess.mean() / excess.std())

    def sortino(self, risk_free: float = 0.05, periods_per_year: int = 8760) -> float:
        r = self._returns()
        if len(r) < 2: return 0.0
        rf_per_bar = risk_free / periods_per_year
        excess     = r - rf_per_bar
        downside   = np.sqrt((excess[excess < 0] ** 2).mean()) if (excess < 0).any() else 1e-9
        return float(np.sqrt(periods_per_year) * excess.mean() / downside)

    def max_drawdown(self) -> float:
        eq = self.equity
        if len(eq) < 2: return 0.0
        roll_max = eq.cummax()
        dd       = (eq - roll_max) / roll_max
        return float(dd.min())

    def calmar(self) -> float:
        mdd = abs(self.max_drawdown())
        if mdd == 0: return 0.0
        tr = (self.equity.iloc[-1] / self.capital - 1)
        return float(tr / mdd)

    def win_rate(self) -> float:
        if len(self.trades) == 0: return 0.0
        return float((self.trades['pnl_net'] > 0).mean())

    def profit_factor(self) -> float:
        wins = self.trades.loc[self.trades['pnl_net'] > 0, 'pnl_net'].sum()
        loss = abs(self.trades.loc[self.trades['pnl_net'] < 0, 'pnl_net'].sum())
        return float(wins / loss) if loss > 0 else float('inf')

    def avg_trade_pnl(self) -> float:
        return float(self.trades['pnl_net'].mean()) if len(self.trades) else 0.0

    def plot(self, show: bool = True) -> go.Figure:
        eq  = self.equity
        trd = self.trades
        fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                            row_heights=[0.5, 0.25, 0.25],
                            subplot_titles=['Equity Curve', 'Drawdown', 'PnL par trade'])

        # Equity
        fig.add_trace(go.Scatter(x=eq.index, y=eq.values, name='Equity',
                                 line=dict(color=ACCENT, width=2)), row=1, col=1)
        fig.add_hline(y=self.capital, line_dash='dot', line_color='gray', row=1, col=1)

        # Drawdown
        roll_max = eq.cummax()
        dd = (eq - roll_max) / roll_max * 100
        fig.add_trace(go.Scatter(x=dd.index, y=dd.values, name='Drawdown %',
                                 fill='tozeroy', fillcolor='rgba(255,75,75,0.2)',
                                 line=dict(color=RED)), row=2, col=1)

        # Trade PnL
        if len(trd):
            colors_t = [ACCENT if p > 0 else RED for p in trd['pnl_net']]
            fig.add_trace(go.Bar(x=trd.index, y=trd['pnl_net'].values,
                                 marker_color=colors_t, name='Trade PnL'), row=3, col=1)

        s = self.summary()
        fig.update_layout(
            title=f"{self.strategy_name} — Sharpe={s['sharpe']:.2f} | "
                  f"Sortino={s['sortino']:.2f} | MaxDD={s['max_drawdown']*100:.1f}% | "
                  f"WinRate={s['win_rate']*100:.1f}%",
            height=700, template='plotly_dark', showlegend=False
        )
        if show: fig.show()
        return fig


print('✅ BacktestResult défini')


In [ ]:
class BacktestEngine:
    """
    Moteur de backtest vectorisé.
    signals_df colonnes requises:
      - entry_time   : datetime (index ou colonne)
      - symbol       : str
      - side         : 'long' | 'short'
      - entry_price  : float
      - tp_price     : float  (0 = pas de TP)
      - sl_price     : float  (0 = pas de SL)
      - pos_usd      : float  (taille notionnelle)
      - exit_time    : datetime (optionnel, time-stop)
    """

    def __init__(self, capital: float = 500.0,
                 taker_fee: float = 0.00035,
                 maker_fee: float = -0.0001,
                 slippage_bps: float = 2.0,
                 entry_type: str = 'taker',
                 exit_type:  str = 'taker'):
        self.capital       = capital
        self.taker_fee     = taker_fee
        self.maker_fee     = maker_fee
        self.slip          = slippage_bps / 10_000
        self.entry_type    = entry_type
        self.exit_type     = exit_type

    def _fee(self, t: str) -> float:
        return self.taker_fee if t == 'taker' else self.maker_fee

    def _round_trip_cost_rate(self) -> float:
        return self._fee(self.entry_type) + self._fee(self.exit_type) + 2 * self.slip

    def run(self, signals: pd.DataFrame, prices: pd.DataFrame = None,
            strategy_name: str = 'Strategy') -> BacktestResult:
        """
        prices: DataFrame index=datetime, columns=symbols, values=close prices
                (nécessaire si exit_time n'est pas fourni dans signals)
        """
        if len(signals) == 0:
            empty = pd.DataFrame(columns=['pnl_gross', 'pnl_net', 'cost'])
            return BacktestResult(empty, pd.Series([self.capital], dtype=float),
                                  self.capital, strategy_name)

        df = signals.copy()
        rt_rate = self._round_trip_cost_rate()

        # Prix de sortie effectifs (avec slippage)
        # Hypothèse: TP/SL atteints à ces prix exacts
        df['cost']      = df['pos_usd'] * rt_rate
        df['pnl_gross'] = df.apply(self._compute_gross, axis=1)
        df['pnl_net']   = df['pnl_gross'] - df['cost']

        # Equity curve
        if 'entry_time' in df.columns:
            df = df.set_index('entry_time')
        df.sort_index(inplace=True)

        cumulative = df['pnl_net'].cumsum() + self.capital
        return BacktestResult(df, cumulative, self.capital, strategy_name)

    @staticmethod
    def _compute_gross(row) -> float:
        side = row.get('side', 'long')
        ep   = row.get('entry_price', 1.0)
        tp   = row.get('tp_price', 0.0)
        sl   = row.get('sl_price', 0.0)
        xp   = row.get('exit_price', 0.0)
        pos  = row.get('pos_usd', 100.0)

        # Utilise exit_price si fourni, sinon tp_price
        exit_p = xp if xp > 0 else (tp if tp > 0 else ep)

        if side == 'long':
            return pos * (exit_p - ep) / ep
        else:  # short
            return pos * (ep - exit_p) / ep


engine = BacktestEngine(capital=CAPITAL_PER_STRAT)
print(f'✅ BacktestEngine prêt | capital=${engine.capital} | '
      f'round-trip cost = {engine._round_trip_cost_rate()*10000:.1f}bps')


## 4. Backtest S8EMS — Market-Making Econophysique

S8EMS est un market-maker basé sur un filtre de Kalman (fair value) et le modèle d'impact de Bouchaud.
On simule les entrées quand le prix s'écarte suffisamment de la fair value.

**Comparaison :** anciens paramètres (spread=1.5bps, pos=$4) vs nouveaux (spread=8bps, pos=$100)


In [ ]:
def kalman_filter_1d(prices: np.ndarray, Q: float = 1e-7, R: float = 1e-5) -> np.ndarray:
    """
    Filtre de Kalman scalaire pour estimer la fair value.
    Retourne le vecteur des fair values estimées.
    """
    n   = len(prices)
    fv  = np.zeros(n)
    P   = 1.0
    x   = prices[0]
    for i in range(n):
        # Prédiction
        P = P + Q
        # Mise à jour
        K = P / (P + R)
        x = x + K * (prices[i] - x)
        P = (1 - K) * P
        fv[i] = x
    return fv


def backtest_s8ems(df_prices: pd.DataFrame,
                   min_spread_bps: float = 8.0,
                   base_notional_pct: float = 0.20,
                   capital: float = 500.0,
                   max_hold_bars: int = 10,
                   Q: float = 1e-7, R: float = 1e-5,
                   engine: BacktestEngine = None) -> BacktestResult:
    """
    Simule S8EMS sur un seul actif (df_prices = série 1h OHLCV).
    Entrée : spread midprice > fair value dépasse min_spread_bps.
    Sortie  : retour à la fair value OU time-stop (max_hold_bars).
    """
    if engine is None:
        engine = BacktestEngine(capital=capital)

    closes = df_prices['close'].values
    highs  = df_prices['high'].values
    lows   = df_prices['low'].values
    times  = df_prices.index
    fv     = kalman_filter_1d(closes, Q=Q, R=R)

    pos_usd = capital * base_notional_pct
    min_spread_frac = min_spread_bps / 10_000

    records = []
    i = 50  # warmup
    while i < len(closes) - max_hold_bars - 1:
        spread_up   = (closes[i] - fv[i]) / fv[i]   # prix > FV → short
        spread_down = (fv[i] - closes[i]) / fv[i]   # prix < FV → long

        trade = None
        if spread_up > min_spread_frac:
            # Short: prix au-dessus de la FV
            ep = closes[i]
            trade = {'side': 'short', 'entry_price': ep, 'fv': fv[i]}
        elif spread_down > min_spread_frac:
            # Long: prix en-dessous de la FV
            ep = closes[i]
            trade = {'side': 'long', 'entry_price': ep, 'fv': fv[i]}

        if trade:
            # Simulation de la sortie sur les barres suivantes
            exit_price = closes[min(i + max_hold_bars, len(closes)-1)]
            # Mean reversion: check if price reverts to FV before time-stop
            for j in range(i+1, min(i + max_hold_bars, len(closes))):
                if trade['side'] == 'short' and closes[j] <= fv[j]:
                    exit_price = closes[j]; break
                elif trade['side'] == 'long'  and closes[j] >= fv[j]:
                    exit_price = closes[j]; break

            trade.update({
                'entry_time': times[i],
                'exit_price': exit_price,
                'tp_price': 0, 'sl_price': 0,
                'pos_usd': pos_usd,
                'symbol': df_prices.columns[0] if hasattr(df_prices, 'columns') else 'BTC',
            })
            records.append(trade)
            i += max_hold_bars  # no overlapping trades
        else:
            i += 1

    if not records:
        empty = pd.DataFrame(columns=['pnl_gross','pnl_net','cost'])
        return BacktestResult(empty, pd.Series([capital]), capital, 'S8EMS')

    signals = pd.DataFrame(records)
    return engine.run(signals, strategy_name='S8EMS')


print('✅ backtest_s8ems() défini')


In [ ]:
btc_1h = RAW['1h']['BTC']
eth_1h = RAW['1h']['ETH']

# === ANCIENS PARAMÈTRES ===
engine_old = BacktestEngine(capital=100.0)
res_old = backtest_s8ems(btc_1h, min_spread_bps=1.5, base_notional_pct=0.04,
                          capital=100.0, engine=engine_old)
s_old = res_old.summary()

# === NOUVEAUX PARAMÈTRES ===
engine_new = BacktestEngine(capital=500.0)
res_new = backtest_s8ems(btc_1h, min_spread_bps=8.0, base_notional_pct=0.20,
                          capital=500.0, engine=engine_new)
s_new = res_new.summary()

print('ANCIENS PARAMÈTRES (spread=1.5bps, pos=4%×$100=$4):')
for k, v in s_old.items():
    print(f'  {k:20s}: {v}')

print('\nNOUVEAUX PARAMÈTRES (spread=8bps, pos=20%×$500=$100):')
for k, v in s_new.items():
    print(f'  {k:20s}: {v}')


In [ ]:
# Courbes d'equity : anciens vs nouveaux paramètres
fig = go.Figure()

eq_old_norm = res_old.equity_curve() / res_old.equity_curve().iloc[0] * 100
eq_new_norm = res_new.equity_curve() / res_new.equity_curve().iloc[0] * 100

fig.add_trace(go.Scatter(
    x=eq_old_norm.index, y=eq_old_norm.values,
    name=f'Anciens params (spread=1.5bps) — Sharpe={s_old["sharpe"]:.2f}',
    line=dict(color=RED, width=2)
))
fig.add_trace(go.Scatter(
    x=eq_new_norm.index, y=eq_new_norm.values,
    name=f'Nouveaux params (spread=8bps) — Sharpe={s_new["sharpe"]:.2f}',
    line=dict(color=ACCENT, width=2)
))
fig.add_hline(y=100, line_dash='dot', line_color='gray')

fig.update_layout(
    title='S8EMS — Equity normalisée (base 100) : Spirale de mort vs Paramètres réparés',
    xaxis_title='Date', yaxis_title='Equity (base 100)',
    height=450, template='plotly_dark'
)
fig.show()

res_new.plot()


## 5. Backtest MomentumLS — Momentum Cross-Sectionnel

Classement des actifs par rendement sur 24h.
**Long** les top-k, **Short** les bottom-k.
Rebalancement horaire.


In [ ]:
def backtest_momentum_ls(data_1h: dict,
                          score_threshold: float = 8.0,
                          top_k: int = 3,
                          bottom_k: int = 3,
                          take_profit_pct: float = 3.0,
                          stop_loss_pct: float = 3.0,
                          max_hold_hours: int = 24,
                          capital: float = 500.0) -> BacktestResult:
    """
    Momentum cross-sectionnel vectorisé.
    data_1h: dict {sym: DataFrame with close column}
    """
    # Aligner tous les actifs sur le même index
    closes = pd.DataFrame({sym: df['close'] for sym, df in data_1h.items()})
    closes = closes.dropna(how='any').sort_index()

    # Rendements sur lookback
    lookback = 24  # barres 1h = 24h
    rets_24h = closes.pct_change(lookback).shift(1)  # shift pour éviter lookahead

    # Score = rendement 24h normalisé × 100
    scores = rets_24h * 100

    pos_usd = capital * 0.20  # 20% du capital par position
    tp_frac = take_profit_pct / 100
    sl_frac = stop_loss_pct / 100

    records = []
    open_pos: dict = {}  # sym → {entry_time, side, entry_price, ...}

    for t_idx in range(lookback + 1, len(closes)):
        t       = closes.index[t_idx]
        row_sc  = scores.iloc[t_idx]
        row_cl  = closes.iloc[t_idx]
        row_prev= closes.iloc[t_idx - 1]

        # Fermer les positions existantes (time-stop ou TP/SL atteint)
        for sym in list(open_pos.keys()):
            pos  = open_pos[sym]
            ep   = pos['entry_price']
            cp   = row_cl[sym]
            held = t_idx - pos['entry_idx']

            pnl_pct = (cp - ep) / ep if pos['side'] == 'long' else (ep - cp) / ep
            do_exit = (pnl_pct >= tp_frac or pnl_pct <= -sl_frac or held >= max_hold_hours)

            if do_exit:
                rec = dict(pos)
                rec['exit_price'] = cp
                rec['entry_time'] = pos['entry_time']
                rec['tp_price']   = 0
                rec['sl_price']   = 0
                records.append(rec)
                del open_pos[sym]

        # Nouveaux signaux
        ranked = row_sc.dropna().sort_values(ascending=False)
        longs  = ranked[ranked > score_threshold].head(top_k).index.tolist()
        shorts = ranked[ranked < -score_threshold].tail(bottom_k).index.tolist()

        for sym in longs:
            if sym not in open_pos and len(open_pos) < top_k + bottom_k:
                open_pos[sym] = {'side': 'long', 'entry_price': row_cl[sym],
                                 'entry_time': t, 'entry_idx': t_idx,
                                 'pos_usd': pos_usd, 'symbol': sym}
        for sym in shorts:
            if sym not in open_pos and len(open_pos) < top_k + bottom_k:
                open_pos[sym] = {'side': 'short', 'entry_price': row_cl[sym],
                                 'entry_time': t, 'entry_idx': t_idx,
                                 'pos_usd': pos_usd, 'symbol': sym}

    if not records:
        empty = pd.DataFrame(columns=['pnl_gross','pnl_net','cost'])
        return BacktestResult(empty, pd.Series([capital]), capital, 'MomentumLS')

    eng = BacktestEngine(capital=capital)
    return eng.run(pd.DataFrame(records), strategy_name='MomentumLS')


print('✅ backtest_momentum_ls() défini')


In [ ]:
# Score threshold=20 (ancien) vs 8 (nouveau)
data_1h = RAW['1h']

res_mom_old = backtest_momentum_ls(data_1h, score_threshold=20, capital=500)
res_mom_new = backtest_momentum_ls(data_1h, score_threshold=8,  capital=500)

s_mo = res_mom_old.summary()
s_mn = res_mom_new.summary()

print('Threshold=20 (ancien):', {k: round(v,3) if isinstance(v,float) else v
                                  for k,v in s_mo.items()})
print('Threshold=8  (nouveau):', {k: round(v,3) if isinstance(v,float) else v
                                   for k,v in s_mn.items()})

# Chart
fig = go.Figure()
fig.add_trace(go.Scatter(x=res_mom_old.equity.index, y=res_mom_old.equity.values,
    name=f'Threshold=20 (Sharpe={s_mo["sharpe"]:.2f})', line=dict(color=YELLOW, width=2)))
fig.add_trace(go.Scatter(x=res_mom_new.equity.index, y=res_mom_new.equity.values,
    name=f'Threshold=8  (Sharpe={s_mn["sharpe"]:.2f})', line=dict(color=ACCENT, width=2)))
fig.add_hline(y=500, line_dash='dot', line_color='gray')
fig.update_layout(title='MomentumLS — Equity Curve', xaxis_title='Date',
                  yaxis_title='Equity ($)', height=420, template='plotly_dark')
fig.show()
res_mom_new.plot()


## 6. Backtest DonchianTrend

Canal de Donchian N=36 barres (15m) + filtre tendance EMA 1h + filtre volume.
TP=2%, SL=1%. Sweep paramétrique sur N et TP.


In [ ]:
def compute_donchian(df: 'pd.DataFrame', n: int = 36) -> 'pd.DataFrame':
    """Calcule les canaux de Donchian."""
    import pandas as pd
    out = df.copy()
    out['dc_upper'] = df['high'].rolling(n).max()
    out['dc_lower'] = df['low'].rolling(n).min()
    out['dc_mid']   = (out['dc_upper'] + out['dc_lower']) / 2
    return out


def backtest_donchian(df_15m: 'pd.DataFrame', df_1h: 'pd.DataFrame',
                       donchian_n: int = 36,
                       ema_period: int = 50,
                       vol_sma: int = 20,
                       vol_mult: float = 1.1,
                       tp_pct: float = 2.0,
                       sl_pct: float = 1.0,
                       max_hold_hours: float = 36.0,
                       capital: float = 500.0) -> 'BacktestResult':
    import pandas as pd, numpy as np
    
    df = compute_donchian(df_15m, donchian_n)
    df['ema_vol']  = df['volume'].rolling(vol_sma).mean()
    df['ema_1h']   = df['close'].resample('1h').last().ffill().reindex(df.index, method='ffill')
    df['ema_trend']= df['ema_1h'].ewm(span=ema_period, adjust=False).mean()
    df = df.dropna()

    tp_frac = tp_pct / 100
    sl_frac = sl_pct / 100
    max_bars = int(max_hold_hours * 4)  # 15m bars per hour
    pos_usd  = capital * 0.20

    records = []
    in_trade = False
    entry_idx = 0

    closes = df['close'].values
    highs  = df['high'].values
    lows   = df['low'].values
    dc_up  = df['dc_upper'].values
    dc_lo  = df['dc_lower'].values
    ema_tr = df['ema_trend'].values
    vols   = df['volume'].values
    evols  = df['ema_vol'].values
    times  = df.index

    i = donchian_n + ema_period
    while i < len(closes) - max_bars - 1:
        if in_trade:
            ep  = entry_ep
            held= i - entry_idx
            if side_t == 'long':
                if highs[i] >= ep * (1 + tp_frac):
                    xp = ep * (1 + tp_frac)
                elif lows[i] <= ep * (1 - sl_frac):
                    xp = ep * (1 - sl_frac)
                elif held >= max_bars:
                    xp = closes[i]
                else:
                    i += 1; continue
            else:
                if lows[i] <= ep * (1 - tp_frac):
                    xp = ep * (1 - tp_frac)
                elif highs[i] >= ep * (1 + sl_frac):
                    xp = ep * (1 + sl_frac)
                elif held >= max_bars:
                    xp = closes[i]
                else:
                    i += 1; continue

            records.append({'entry_time': times[entry_idx], 'side': side_t,
                            'entry_price': ep, 'exit_price': xp,
                            'tp_price': 0, 'sl_price': 0, 'pos_usd': pos_usd})
            in_trade = False; i += 1; continue

        # Signal long: breakout haut + tendance haussière + volume
        prev_close = closes[i-1]
        trend_up   = ema_tr[i] > ema_tr[i - 4]   # EMA montante sur 4 barres
        vol_ok     = vols[i] > evols[i] * vol_mult

        if prev_close <= dc_up[i-1] and closes[i] > dc_up[i] and trend_up and vol_ok:
            in_trade = True; side_t = 'long'
            entry_ep = closes[i]; entry_idx = i
        elif prev_close >= dc_lo[i-1] and closes[i] < dc_lo[i] and not trend_up and vol_ok:
            in_trade = True; side_t = 'short'
            entry_ep = closes[i]; entry_idx = i
        i += 1

    if not records:
        import pandas as pd
        empty = pd.DataFrame(columns=['pnl_gross','pnl_net','cost'])
        return BacktestResult(empty, pd.Series([capital]), capital, 'DonchianTrend')

    import pandas as pd
    eng = BacktestEngine(capital=capital)
    return eng.run(pd.DataFrame(records), strategy_name='DonchianTrend')


print('✅ backtest_donchian() défini')


In [ ]:
import pandas as pd, numpy as np

btc_15m = RAW['15m']['BTC']
btc_1h  = RAW['1h']['BTC']

# Backtest avec paramètres calibrés
res_don = backtest_donchian(btc_15m, btc_1h, donchian_n=36, tp_pct=2.0, sl_pct=1.0, capital=500)
s_don   = res_don.summary()
print('DonchianTrend (N=36, TP=2%, SL=1%):')
for k, v in s_don.items():
    print(f'  {k:20s}: {v}')

res_don.plot()


In [ ]:
# Sweep paramétrique — donchian_n × tp_pct
import pandas as pd
ns   = [24, 36, 48]
tps  = [1.0, 1.5, 2.0, 2.5, 3.0]
results_grid = []

for n in ns:
    for tp in tps:
        r = backtest_donchian(btc_15m, btc_1h, donchian_n=n, tp_pct=tp, sl_pct=1.0, capital=500)
        s = r.summary()
        results_grid.append({'donchian_n': n, 'tp_pct': tp,
                             'sharpe': round(s['sharpe'],3),
                             'total_return': round(s['total_return'],2),
                             'win_rate': round(s['win_rate']*100,1),
                             'n_trades': s['n_trades'],
                             'max_dd': round(s['max_drawdown']*100,2)})

grid_df = pd.DataFrame(results_grid)
print('Sweep Donchian:')
print(grid_df.sort_values('sharpe', ascending=False).to_string(index=False))

best = grid_df.sort_values('sharpe', ascending=False).iloc[0]
print(f'\n✅ Meilleurs params: N={int(best.donchian_n)}, TP={best.tp_pct}%  (Sharpe={best.sharpe})')


## 7. Backtest RSIBollingerReversion

RSI(14) < 35 (survente) + re-entrée dans les Bandes de Bollinger.
Confirmation sur 2 barres pour éviter les faux signaux.


In [ ]:
def compute_rsi(series: 'pd.Series', period: int = 14) -> 'pd.Series':
    import pandas as pd, numpy as np
    delta = series.diff()
    gain  = delta.clip(lower=0).ewm(alpha=1/period, adjust=False).mean()
    loss  = (-delta.clip(upper=0)).ewm(alpha=1/period, adjust=False).mean()
    rs    = gain / loss.replace(0, 1e-9)
    return 100 - 100 / (1 + rs)


def compute_bollinger(series: 'pd.Series', period: int = 20, k: float = 2.0):
    mid = series.rolling(period).mean()
    std = series.rolling(period).std()
    return mid - k * std, mid, mid + k * std


def backtest_rsi_bollinger(df_15m: 'pd.DataFrame',
                            rsi_oversold: float = 35.0,
                            zscore_entry: float = -1.5,
                            rsi_period: int = 14,
                            bb_period: int = 20,
                            bb_k: float = 2.0,
                            tp_pct: float = 1.5,
                            sl_pct: float = 0.8,
                            time_stop_bars: int = 16,
                            capital: float = 500.0) -> 'BacktestResult':
    import pandas as pd, numpy as np
    
    df        = df_15m.copy()
    df['rsi'] = compute_rsi(df['close'], rsi_period)
    bb_lo, bb_mid, bb_up = compute_bollinger(df['close'], bb_period, bb_k)
    df['bb_lo'] = bb_lo
    df['bb_mid']= bb_mid
    df['bb_up'] = bb_up

    # Z-score du prix relatif aux BB
    df['zscore'] = (df['close'] - bb_mid) / (bb_up - bb_lo + 1e-9) * 4
    df = df.dropna()

    pos_usd  = capital * 0.20
    tp_frac  = tp_pct / 100
    sl_frac  = sl_pct / 100

    closes  = df['close'].values
    highs   = df['high'].values
    lows    = df['low'].values
    rsi_v   = df['rsi'].values
    zscore  = df['zscore'].values
    bb_lo_v = df['bb_lo'].values
    times   = df.index

    records  = []
    in_trade = False

    for i in range(2, len(closes) - time_stop_bars - 1):
        if in_trade:
            ep   = entry_ep
            held = i - entry_idx
            # Long trade
            if highs[i] >= ep * (1 + tp_frac):
                xp = ep * (1 + tp_frac)
            elif lows[i] <= ep * (1 - sl_frac):
                xp = ep * (1 - sl_frac)
            elif held >= time_stop_bars:
                xp = closes[i]
            else:
                continue

            records.append({'entry_time': times[entry_idx], 'side': 'long',
                            'entry_price': ep, 'exit_price': xp,
                            'tp_price': 0, 'sl_price': 0, 'pos_usd': pos_usd})
            in_trade = False
            continue

        # Condition d'entrée : barre t-1 oversold ET barre t re-entre dans BB
        bar_t1_oversold  = (rsi_v[i-1] < rsi_oversold) and (zscore[i-1] < zscore_entry)
        bar_t_reentry    = closes[i] > bb_lo_v[i]   # re-entrée dans la bande

        if bar_t1_oversold and bar_t_reentry and not in_trade:
            in_trade  = True
            entry_ep  = closes[i]
            entry_idx = i

    if not records:
        import pandas as pd
        empty = pd.DataFrame(columns=['pnl_gross','pnl_net','cost'])
        return BacktestResult(empty, pd.Series([capital]), capital, 'RSIBollingerReversion')

    import pandas as pd
    eng = BacktestEngine(capital=capital)
    return eng.run(pd.DataFrame(records), strategy_name='RSIBollingerReversion')


print('✅ backtest_rsi_bollinger() défini')


In [ ]:
import pandas as pd

btc_15m = RAW['15m']['BTC']

# Comparaison rsi_oversold=30 vs 35, zscore=-2.0 vs -1.5
combos = [
    (30, -2.0, 'RSI<30, z<-2.0 (strict)'),
    (35, -2.0, 'RSI<35, z<-2.0'),
    (35, -1.5, 'RSI<35, z<-1.5 (nouveau)'),
    (40, -1.5, 'RSI<40, z<-1.5 (large)'),
]

fig = go.Figure()
rsi_results = {}
for rsi_th, z_th, label in combos:
    r = backtest_rsi_bollinger(btc_15m, rsi_oversold=rsi_th, zscore_entry=z_th, capital=500)
    s = r.summary()
    rsi_results[label] = r
    print(f'{label}: Sharpe={s["sharpe"]:.2f} | WR={s["win_rate"]*100:.1f}% | '
          f'PF={s["profit_factor"]:.2f} | n={s["n_trades"]} | '
          f'Return={s["total_return"]:.1f}%')
    fig.add_trace(go.Scatter(x=r.equity.index, y=r.equity.values, name=label))

fig.add_hline(y=500, line_dash='dot', line_color='gray')
fig.update_layout(title='RSIBollingerReversion — Comparaison paramètres',
                  xaxis_title='Date', yaxis_title='Equity ($)',
                  height=420, template='plotly_dark')
fig.show()

# Plot du meilleur
best_rsi = rsi_results['RSI<35, z<-1.5 (nouveau)']
best_rsi.plot()


## 8. Backtest MeanReversionKalman

Filtre de Kalman sur le prix, z-score des résidus.
Entrée quand |z| > 0.8, sortie quand |z| < 0.0.


In [ ]:
def backtest_mrk(df_prices: 'pd.DataFrame',
                  z_entry: float = 0.8,
                  z_exit: float = 0.0,
                  z_stop: float = 4.5,
                  max_hold_bars: int = 16,  # 240min / 15m
                  Q: float = 1e-6,
                  R: float = 1e-4,
                  capital: float = 500.0) -> 'BacktestResult':
    import pandas as pd, numpy as np
    
    closes = df_prices['close'].values
    times  = df_prices.index
    n      = len(closes)
    fv     = kalman_filter_1d(closes, Q=Q, R=R)

    # Z-score des résidus
    residuals = closes - fv
    roll_win  = 60  # fenêtre rolling pour std
    res_std   = pd.Series(residuals).rolling(roll_win).std().values
    res_std   = np.where(res_std < 1e-9, 1e-9, res_std)
    zscores   = residuals / res_std

    pos_usd  = capital * 0.20
    records  = []
    in_trade = False

    i = roll_win + 10
    while i < n - max_hold_bars - 1:
        z = zscores[i]
        if np.isnan(z):
            i += 1; continue

        if in_trade:
            held = i - entry_idx
            zc   = zscores[i]
            do_exit = False
            if trade_side == 'long'  and zc >= z_exit:  do_exit = True; xp = closes[i]
            if trade_side == 'short' and zc <= -z_exit: do_exit = True; xp = closes[i]
            if abs(zc) > z_stop:                        do_exit = True; xp = closes[i]
            if held >= max_hold_bars:                   do_exit = True; xp = closes[i]

            if do_exit:
                records.append({'entry_time': times[entry_idx], 'side': trade_side,
                                'entry_price': entry_ep, 'exit_price': xp,
                                'tp_price': 0, 'sl_price': 0, 'pos_usd': pos_usd})
                in_trade = False
            i += 1; continue

        if z < -z_entry:
            in_trade = True; trade_side = 'long'
            entry_ep  = closes[i]; entry_idx = i
        elif z > z_entry:
            in_trade = True; trade_side = 'short'
            entry_ep  = closes[i]; entry_idx = i
        i += 1

    if not records:
        import pandas as pd
        empty = pd.DataFrame(columns=['pnl_gross','pnl_net','cost'])
        return BacktestResult(empty, pd.Series([capital]), capital, 'MeanReversionKalman')

    import pandas as pd
    eng = BacktestEngine(capital=capital)
    return eng.run(pd.DataFrame(records), strategy_name='MeanReversionKalman')


print('✅ backtest_mrk() défini')


In [ ]:
import pandas as pd

btc_15m = RAW['15m']['BTC']

res_mrk_10 = backtest_mrk(btc_15m, z_entry=1.0, capital=500)
res_mrk_08 = backtest_mrk(btc_15m, z_entry=0.8, capital=500)

s10 = res_mrk_10.summary()
s08 = res_mrk_08.summary()
print(f'z_entry=1.0: Sharpe={s10["sharpe"]:.2f} | WR={s10["win_rate"]*100:.1f}% | n={s10["n_trades"]}')
print(f'z_entry=0.8: Sharpe={s08["sharpe"]:.2f} | WR={s08["win_rate"]*100:.1f}% | n={s08["n_trades"]}')

fig = go.Figure()
fig.add_trace(go.Scatter(x=res_mrk_10.equity.index, y=res_mrk_10.equity.values,
    name=f'z_entry=1.0 (Sharpe={s10["sharpe"]:.2f})', line=dict(color=YELLOW)))
fig.add_trace(go.Scatter(x=res_mrk_08.equity.index, y=res_mrk_08.equity.values,
    name=f'z_entry=0.8 (Sharpe={s08["sharpe"]:.2f})', line=dict(color=ACCENT)))
fig.add_hline(y=500, line_dash='dot', line_color='gray')
fig.update_layout(title='MeanReversionKalman — z_entry=1.0 vs 0.8',
                  xaxis_title='Date', yaxis_title='Equity ($)',
                  height=420, template='plotly_dark')
fig.show()
res_mrk_08.plot()


## 9. Dashboard comparatif — Toutes les stratégies

Comparaison côte à côte de toutes les stratégies avec les paramètres calibrés.


In [ ]:
import pandas as pd
# Récupérer tous les résultats avec paramètres finaux
all_results = {
    'S8EMS':               res_new,
    'MomentumLS':          res_mom_new,
    'DonchianTrend':       res_don,
    'RSIBollingerRev':     best_rsi,
    'MeanRevKalman':       res_mrk_08,
}

# Tableau de métriques
rows = []
for name, res in all_results.items():
    s = res.summary()
    rows.append({
        'Stratégie':    name,
        'Sharpe':       round(s['sharpe'], 2),
        'Sortino':      round(s['sortino'], 2),
        'Calmar':       round(s['calmar'], 2),
        'MaxDD%':       round(s['max_drawdown'] * 100, 1),
        'WinRate%':     round(s['win_rate'] * 100, 1),
        'ProfitFactor': round(s['profit_factor'], 2),
        'AvgTrade$':    round(s['avg_trade_pnl'], 3),
        'Trades':       s['n_trades'],
        'Return%':      round(s['total_return'], 1),
    })

metrics_df = pd.DataFrame(rows).set_index('Stratégie')
print('=== TABLEAU DES MÉTRIQUES — PARAMÈTRES CALIBRÉS ===')
print(metrics_df.to_string())


In [ ]:
# Graphique: Equity curves côte à côte
fig = make_subplots(
    rows=3, cols=2, shared_xaxes=False,
    subplot_titles=list(all_results.keys()) + [''],
    vertical_spacing=0.12
)

color_map = [ACCENT, '#f1c40f', '#e74c3c', '#9b59b6', '#3498db']
positions  = [(1,1), (1,2), (2,1), (2,2), (3,1)]

for idx, (name, res) in enumerate(all_results.items()):
    r, c = positions[idx]
    eq   = res.equity_curve()
    s    = res.summary()
    fig.add_trace(
        go.Scatter(x=eq.index, y=eq.values, name=name,
                   line=dict(color=color_map[idx], width=1.5),
                   showlegend=True),
        row=r, col=c
    )
    fig.add_hline(y=500, line_dash='dot', line_color='rgba(255,255,255,0.2)',
                  row=r, col=c)

fig.update_layout(
    title='Artemisia v9 — Equity Curves par stratégie (paramètres calibrés)',
    height=900, template='plotly_dark'
)
fig.show()


In [ ]:
# Corrélation des rendements
import pandas as pd, numpy as np

all_rets = {}
for name, res in all_results.items():
    eq = res.equity_curve()
    # Rééchantillonner sur une grille horaire commune
    eq_h = eq.resample('1h').last().ffill()
    all_rets[name] = eq_h.pct_change().dropna()

ret_df   = pd.DataFrame(all_rets).dropna()
corr_mat = ret_df.corr()

fig = go.Figure(data=go.Heatmap(
    z=corr_mat.values,
    x=corr_mat.columns.tolist(),
    y=corr_mat.index.tolist(),
    colorscale='RdBu', zmid=0,
    text=corr_mat.round(2).values,
    texttemplate='%{text}',
    showscale=True
))
fig.update_layout(title='Corrélation des rendements entre stratégies',
                  height=450, template='plotly_dark')
fig.show()

print('\nStratégies peu corrélées = diversification efficace.')
print('Corrélation moyenne:', round(corr_mat.where(corr_mat < 1).stack().mean(), 3))


In [ ]:
# Feux de signalisation — Quelles stratégies valent la peine?
import pandas as pd

def traffic_light(row):
    score = 0
    if row['Sharpe']  >= 1.5: score += 2
    elif row['Sharpe'] >= 0.8: score += 1
    if row['WinRate%'] >= 55: score += 1
    if row['ProfitFactor'] >= 1.5: score += 1
    if row['MaxDD%'] >= -20: score += 1
    if score >= 4: return '🟢 Deploy'
    elif score >= 2: return '🟡 Améliorer'
    else: return '🔴 Suspendre'

metrics_df['Verdict'] = metrics_df.apply(traffic_light, axis=1)
print('=== VERDICT PAR STRATÉGIE ===')
print(metrics_df[['Sharpe','WinRate%','ProfitFactor','MaxDD%','Return%','Verdict']].to_string())


## 10. Validation Walk-Forward

Diviser les données en fenêtres glissantes pour détecter le sur-ajustement.
**Train :** 120 jours → **Test :** 30 jours → avancer de 30 jours.


In [ ]:
import pandas as pd, numpy as np

def walk_forward_donchian(df_15m: pd.DataFrame, df_1h: pd.DataFrame,
                           train_days: int = 120, test_days: int = 30,
                           step_days: int = 30, capital: float = 500.0):
    """
    Walk-forward sur DonchianTrend.
    Phase IN-SAMPLE : optimise donchian_n et tp_pct.
    Phase OUT-OF-SAMPLE : applique les meilleurs paramètres.
    """
    total_days = (df_15m.index[-1] - df_15m.index[0]).days
    starts     = range(0, total_days - train_days - test_days, step_days)
    ns   = [24, 36, 48]
    tps  = [1.5, 2.0, 2.5]

    wf_records = []

    for start_d in starts:
        t0      = df_15m.index[0] + pd.Timedelta(days=start_d)
        t_split = t0 + pd.Timedelta(days=train_days)
        t_end   = t_split + pd.Timedelta(days=test_days)

        train_15 = df_15m.loc[(df_15m.index >= t0) & (df_15m.index < t_split)]
        train_1h = df_1h.loc[(df_1h.index  >= t0) & (df_1h.index  < t_split)]
        test_15  = df_15m.loc[(df_15m.index >= t_split) & (df_15m.index < t_end)]
        test_1h  = df_1h.loc[(df_1h.index   >= t_split) & (df_1h.index  < t_end)]

        if len(train_15) < 500 or len(test_15) < 100:
            continue

        # IN-SAMPLE: trouver les meilleurs paramètres
        best_sh, best_n, best_tp = -999, 36, 2.0
        for n in ns:
            for tp in tps:
                try:
                    r = backtest_donchian(train_15, train_1h, donchian_n=n,
                                         tp_pct=tp, sl_pct=1.0, capital=capital)
                    sh = r.sharpe()
                    if sh > best_sh:
                        best_sh = sh; best_n = n; best_tp = tp
                except: pass

        # OUT-OF-SAMPLE: appliquer les meilleurs paramètres
        try:
            r_oos = backtest_donchian(test_15, test_1h, donchian_n=best_n,
                                      tp_pct=best_tp, sl_pct=1.0, capital=capital)
            sh_oos = r_oos.sharpe()
        except:
            sh_oos = float('nan')

        wf_records.append({
            'period_start':  t0.date(),
            'train_end':     t_split.date(),
            'test_end':      t_end.date(),
            'best_n':        best_n,
            'best_tp':       best_tp,
            'sharpe_IS':     round(best_sh, 3),
            'sharpe_OOS':    round(sh_oos, 3),
        })
        print(f'  Fenêtre {t0.date()}→{t_end.date()} | '
              f'Best: N={best_n}, TP={best_tp}% | IS Sharpe={best_sh:.2f} | OOS Sharpe={sh_oos:.2f}')

    return pd.DataFrame(wf_records)


print('Lancement walk-forward DonchianTrend...')
wf_df = walk_forward_donchian(RAW['15m']['BTC'], RAW['1h']['BTC'])
print('\nRésultats walk-forward:')
print(wf_df.to_string(index=False))


In [ ]:
import pandas as pd, numpy as np

if len(wf_df) > 0:
    avg_is  = wf_df['sharpe_IS'].mean()
    avg_oos = wf_df['sharpe_OOS'].mean()
    decay   = (avg_is - avg_oos) / avg_is * 100 if avg_is != 0 else 0

    print(f'Sharpe IS moyen  : {avg_is:.3f}')
    print(f'Sharpe OOS moyen : {avg_oos:.3f}')
    print(f'Dégradation      : {decay:.1f}% — ', end='')
    if decay < 30:
        print('✅ Pas de sur-ajustement majeur')
    elif decay < 60:
        print('🟡 Sur-ajustement modéré — surveiller')
    else:
        print('🔴 Sur-ajustement sévère — paramètres trop optimisés')

    fig = go.Figure()
    x = list(range(len(wf_df)))
    fig.add_trace(go.Bar(x=x, y=wf_df['sharpe_IS'], name='Sharpe In-Sample', marker_color=ACCENT))
    fig.add_trace(go.Bar(x=x, y=wf_df['sharpe_OOS'], name='Sharpe Out-of-Sample', marker_color=YELLOW))
    fig.add_hline(y=0, line_color='gray')
    fig.update_layout(
        title='Walk-Forward DonchianTrend — IS vs OOS Sharpe (risque de sur-ajustement)',
        xaxis_title='Fenêtre', yaxis_title='Sharpe Ratio',
        barmode='group', height=420, template='plotly_dark'
    )
    fig.show()


## 11. Critère de Kelly — Sizing Optimal

**Full Kelly** : fraction optimale théorique (trop agressive en pratique)
**Half Kelly** : recommandé — réduit le drawdown de ~50% sans sacrifice majeur sur les rendements


In [ ]:
import pandas as pd, numpy as np

def compute_kelly(trades: pd.DataFrame, capital: float = 500.0,
                  kelly_fraction: float = 0.5) -> dict:
    """
    Calcule le full Kelly et le Kelly fractionné.
    Trades doit avoir une colonne 'pnl_net'.
    """
    if len(trades) < 5:
        return {'full_kelly_pct': 0, 'half_kelly_pct': 0, 'optimal_pos_usd': 0}

    wins = trades.loc[trades['pnl_net'] > 0, 'pnl_net']
    loss = trades.loc[trades['pnl_net'] < 0, 'pnl_net']

    p     = len(wins) / len(trades)       # probabilité de gain
    q     = 1 - p                         # probabilité de perte

    avg_w = wins.mean() if len(wins) else 0
    avg_l = abs(loss.mean()) if len(loss) else 1e-9

    b = avg_w / avg_l  # ratio gain/perte moyen

    if b <= 0 or avg_l <= 0:
        return {'full_kelly_pct': 0, 'half_kelly_pct': 0, 'optimal_pos_usd': 0,
                'win_prob': p, 'payoff_ratio': b}

    full_kelly = (p * b - q) / b  # formule de Kelly
    full_kelly = max(0, min(full_kelly, 0.25))  # plafonner à 25%
    half_kelly = full_kelly * kelly_fraction

    return {
        'win_prob':          round(p, 4),
        'payoff_ratio':      round(b, 4),
        'full_kelly_pct':    round(full_kelly * 100, 2),
        'half_kelly_pct':    round(half_kelly * 100, 2),
        'optimal_pos_usd':   round(capital * half_kelly, 2),
    }


print('=== CRITÈRE DE KELLY PAR STRATÉGIE (Half-Kelly recommandé) ===')
print(f'{"Stratégie":<22} {"P(win)":>8} {"Payoff":>8} {"FullK%":>8} {"HalfK%":>8} {"Pos $":>9}')
print('-' * 70)

kelly_results = {}
for name, res in all_results.items():
    trd = res.trade_log()
    if 'pnl_net' not in trd.columns or len(trd) == 0:
        continue
    k = compute_kelly(trd, capital=500)
    kelly_results[name] = k
    print(f'{name:<22} {k["win_prob"]:>8.3f} {k["payoff_ratio"]:>8.3f} '
          f'{k["full_kelly_pct"]:>8.2f} {k["half_kelly_pct"]:>8.2f} '
          f'{k["optimal_pos_usd"]:>9.2f}')

print('\n⚠️  Full Kelly peut provoquer des drawdowns de -50%+ — toujours utiliser Half-Kelly ou moins.')


## 12. Conclusions & Plan d'action

### Diagnostic des résultats calamiteux

**Cause racine :** les positions étaient mathématiquement impossibles à rentabiliser.

| Paramètre | Ancien | Nouveau | Impact |
|-----------|--------|---------|--------|
| Capital S8EMS | $100 | $500 | +400% de puissance |
| Position S8EMS | $4 (4%) | $100 (20%) | ×25 le PnL brut |
| Spread min S8EMS | 1.5 bps | 8.0 bps | Au-dessus du break-even |
| Score MomentumLS | 20 | 8 | ×3 les signaux |
| Fees round-trip | 11 bps | 11 bps | inchangé |

**Formule clé :** `PnL_net = position × (spread - round_trip_cost_bps/10000)`
Avec l'ancien setup : `$4 × (1.5 - 11) / 10000 = -$0.0038` par trade → **perte garantie à chaque trade**.

### Recommandations

1. **S8EMS** : validé mathématiquement avec nouveaux paramètres. Surveiller la fréquence réelle.
2. **MomentumLS** : dépend fortement de la volatilité du marché. Réduire score_threshold à 8.
3. **DonchianTrend** : meilleure stratégie de tendance. Paramètre robuste dans le walk-forward.
4. **RSIBollingerReversion** : bon en marché range, fragile en tendance forte.
5. **MeanRevKalman** : cohérent mais peu de trades — z_entry=0.8 bon compromis.

### Plan d'action

1. **Semaine 1-2 :** Paper trading avec nouveaux paramètres (déjà dans config_v9.json)
2. **Semaine 3 :** Comparer les métriques live vs backtest
3. **Mois 2 :** Si Sharpe live > 0.5 → passer en vrai capital avec $500/strat
4. **Continu :** Re-runner ce backtest chaque mois avec nouvelles données


In [ ]:
import pandas as pd, numpy as np

# Projection mensuelle espérée (à partir des résultats backtest)
print('=== PROJECTION PnL MENSUEL (extrapolation backtest 180j → 30j) ===')
print(f'{"Stratégie":<22} {"PnL total$":>12} {"PnL/mois$":>12} {"Sharpe":>8} {"MaxDD%":>8}')
print('-' * 65)

for name, res in all_results.items():
    s = res.summary()
    monthly_pnl = s['total_pnl'] / 6.0  # 180j / 6 = 30j
    print(f'{name:<22} {s["total_pnl"]:>12.2f} {monthly_pnl:>12.2f} '
          f'{s["sharpe"]:>8.2f} {s["max_drawdown"]*100:>8.1f}')

total_monthly = sum(res.summary()['total_pnl'] for res in all_results.values()) / 6.0
print(f'{"TOTAL PORTEFEUILLE":<22} {"":>12} {total_monthly:>12.2f}')
print()
print('⚠️  Ces projections sont INDICATIVES. Les performances passées ne garantissent pas les résultats futurs.')
print('✅ Validation paper trading minimum 2 semaines avant de déployer du capital réel.')


---
## Annexe — Résumé des paramètres recommandés

Ces paramètres sont déjà appliqués dans `config_v9.json` :

```json
S8EMS          : min_spread_bps=8.0, base_notional_pct=0.20, max_hold_s=600
MomentumLS     : score_threshold=8,  take_profit_pct=3%,     max_hold_hours=24
BreakoutCtrl   : vr_min=0.8,         take_profit_pct=4%,     max_hold_hours=20
MeanRevKalman  : z_entry=0.8,        z_exit=0.0,             max_hold_minutes=240
DonchianTrend  : donchian_n=36,      stop_loss_pct=1%,       take_profit_pct=2%
RSIBollinger   : rsi_oversold=35,    zscore_entry=-1.5,      take_profit_pct=1.5%
```

**Break-even minimum (Hyperliquid) :** spread ≥ **11 bps** pour les trades taker+taker.

*Généré par Artemisia v9 Backtest Engine — 2026*
